In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


# seaborn for visualisation
import seaborn as sns
sns.set_context("talk")

# seaborn has some unhelpful warnings at the moment
import warnings
warnings.filterwarnings("ignore", module="seaborn")

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score

def plot_decision_tree(tree_model):
    fig, ax = plt.subplots(figsize=(20,8))
    plot_tree(tree_model,  
        filled=True, 
        impurity=False, 
        feature_names=input_features, 
        class_names=["No","Yes"], 
        proportion=True, 
        ax=ax)
    plt.show()

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

: 

In [ ]:
team_data = pd.read_csv('/kaggle/input/datasets/swaptr/fifa-wc-2026-teams/teams.csv')

round_of_16 = {
    "Argentina", "Belgium", "Brazil", "Colombia",
    "England", "France", "Morocco", "Norway",
    "Portugal","Spain","Switzerland","United States",
    "Canada","Paraguay","Mexico","Egypt"
}

team_data["made_r16"] = team_data["team_country"].isin(round_of_16)

team_data

In [ ]:
refined_team_data = team_data[["team_country","made_r16","avg_age","players_used","cards_yellow",'offsides', 'crosses', 'interceptions',"possession","shots_on_target_pct","shots_per90","gk_save_pct","gk_pens_save_pct","goals_per90","assists_per90"]].copy()
refined_team_data

In [ ]:
factors = ["avg_age","players_used","cards_yellow",'offsides', 'crosses', 'interceptions',"possession","shots_on_target_pct","shots_per90","gk_save_pct","gk_pens_save_pct","goals_per90","assists_per90"]

refined_team_data.groupby('made_r16')[factors].mean().round(2)

In [ ]:
sns.catplot(data=refined_team_data, kind='box', y='possession', x="made_r16", aspect=2, height=5);

the outlier for made r16 with just 30% possession was Paraguay. but perhaps not shocking given they won vs. germany in r32 on penalties

In [ ]:
sns.catplot(data=refined_team_data, kind='box', y='gk_save_pct', x="made_r16", aspect=2, height=5);

interestingly the outlier for goalkeeper save % that made it to r16 is argentina! shows you how heavy they are on attack and messi.

In [ ]:
sns.displot(data=refined_team_data, kind='kde', x='offsides', hue='made_r16', aspect=2); 

In [ ]:
sns.relplot(data=refined_team_data, kind='scatter', x='possession', y='crosses',hue="made_r16", alpha=1, aspect =2); 

In [ ]:
sns.relplot(data=refined_team_data, kind='scatter', y='shots_per90', x='shots_on_target_pct',hue="made_r16", alpha=1, aspect =2); 

In [ ]:
sns.relplot(data=refined_team_data, kind='scatter', y='shots_per90', x='possession',hue="made_r16", alpha=1, aspect =2); 

possession is emerging as a super key factor; correlations w/ almost everything else; have the ball and the rest tends to follow, right?

In [ ]:
input_features = ["avg_age","players_used","cards_yellow",'offsides', 'crosses', 'interceptions',"possession","shots_on_target_pct","shots_per90","gk_save_pct","gk_pens_save_pct","goals_per90","assists_per90"]
X = refined_team_data[input_features]
y = refined_team_data['made_r16']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.65, random_state=1)
# create the model and display the decision tree
tree_model = DecisionTreeClassifier(max_depth=4).fit(X_train, y_train)
plot_decision_tree(tree_model)
# create a list of the predictions and calculate the metrics and confusion matrix
y_pred = tree_model.predict(X_test)
print("Precision: ",round(100*precision_score(y_test, y_pred, zero_division=0),1),"%")
print("Recall: ",round(100*recall_score(y_test, y_pred),1),"%")
pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted'], margins=True)

In [ ]:
input_features = ["possession","shots_on_target_pct","gk_save_pct","gk_pens_save_pct","goals_per90"]
X = refined_team_data[input_features]
y = refined_team_data['made_r16']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.65, random_state=1)
# create the model and display the decision tree
tree_model = DecisionTreeClassifier(max_depth=4).fit(X_train, y_train)
plot_decision_tree(tree_model)
# create a list of the predictions and calculate the metrics and confusion matrix
y_pred = tree_model.predict(X_test)
print("Precision: ",round(100*precision_score(y_test, y_pred, zero_division=0),1),"%")
print("Recall: ",round(100*recall_score(y_test, y_pred),1),"%")
pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted'], margins=True)

In [ ]:
input_features = ["possession","shots_on_target_pct","gk_save_pct"]
X = refined_team_data[input_features]
y = refined_team_data['made_r16']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.65, random_state=1)
# create the model and display the decision tree
tree_model = DecisionTreeClassifier(max_depth=4).fit(X_train, y_train)
plot_decision_tree(tree_model)
# create a list of the predictions and calculate the metrics and confusion matrix
y_pred = tree_model.predict(X_test)
print("Precision: ",round(100*precision_score(y_test, y_pred, zero_division=0),1),"%")
print("Recall: ",round(100*recall_score(y_test, y_pred),1),"%")
pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted'], margins=True)

In [ ]:
# Define the input features, X, and the target feature, y, perform the training-testing split
input_features = ["shots_per90","crosses"]
X = refined_team_data[input_features]
y = refined_team_data['possession']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.65, random_state=1)

# create the model and display the parameters - output the coefficients and y-intercept
linear_model = LinearRegression().fit(X_train,y_train)
print('Coefficients: \n', linear_model.coef_.round(3))
print('Intercept: \n', linear_model.intercept_.round(3))

# create a list of the predictions and calculate the metrics (RMSE and R² score) by to the target values in y_test
y_pred = linear_model.predict(X_test) 
print('RMSE: ',round(mean_squared_error(y_test, y_pred),3))
print('R²: ', round(100*r2_score(y_test, y_pred),3), "%")

In [ ]:
input_features = ["possession","crosses","shots_per90"]
X = refined_team_data[input_features]
y = refined_team_data['goals_per90']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.65, random_state=1)

# create the model and display the parameters - output the coefficients and y-intercept
linear_model = LinearRegression().fit(X_train,y_train)
print('Coefficients: \n', linear_model.coef_.round(3))
print('Intercept: \n', linear_model.intercept_.round(3))

# create a list of the predictions and calculate the metrics (RMSE and R² score) by to the target values in y_test
y_pred = linear_model.predict(X_test) 
print('RMSE: ',round(mean_squared_error(y_test, y_pred),3))
print('R²: ', round(100*r2_score(y_test, y_pred),3), "%")

Evaluation
You obviously can’t predict everything about how successful a team will be from numbers. Things like:
* Player experience in tier 1 club football
* Manager/coaching staff
* Injuries
* All have impacts we can’t quantify.

What’s more, the dataset isn’t perfect.
* It ignores historic performance, standout performances, unexpected losses
* Is a really quite small sample; again we ought to mix in the past 2 or 3 world cups too to get something more representative. This would be a good future project.
* I picked out a select few factors I thought might have worked as else this table had 136 columns. So I might’ve missed a few – clean sheet % immediately comes to mind!